# Signpost-D, part 2: evaluation through vLLM

Scores models exactly as the Typed Decisions benchmark does: the model is served with **vLLM** (`vllm serve`, the benchmark scorer's flags) and asked through `POST /v1/systemone`, one request per case with all its questions, and the answers are scored by a copy of the benchmark scorer's own `score_one`. For each entry in `RUNS` it:

1. builds the model: Qwen3.5-4B with the run's LoRA weight changes merged in, saved as a **full** Qwen3.5 checkpoint (vision tower and processor files included), because vLLM's `/v1/systemone` serves only listed architectures (`Qwen3_5ForConditionalGeneration` is one; a text-only export is not);
2. starts vLLM on it and checks that vLLM's prompt has exactly as many tokens as the one `01_train.ipynb` trains on;
3. scores the benchmark's `train` split (never trained on), which compares the runs, and, for the runs marked, its `test` split: accuracy, KL from the gold, Brier and 10-bin ECE, by type and question, plus vLLM's own diagnostics (how much probability went to the labels) and the time per case.

The runs are the untrained base, the three seeds from `01_train.ipynb`, and their soup at alpha 1.0 and 0.75 (the final model). With `PUBLISH`, the final model is built again and uploaded. Needs one GPU; results go to `checkpoints/evaluation-vllm/` and, with `CKPT_REPO`, to that private repo.


## Parameters

Which runs to score and the server's flags (the benchmark scorer's defaults, unchanged). The `test` split is scored only for runs marked `"benchmark": True`; runs scored before are read back, never scored twice.


In [ ]:
MODEL = {"name": "Qwen/Qwen3.5-4B"}
DATASET = "skamalj/signpost-d-decisions"   # our dev records, for the prompt check
SEEDS = [42, 1042, 2042]
RUN = "signpost-d-Qwen3.5-4B/lr0.0001-r16-ep2-seed{seed}/attempt-seed{seed}/epoch2_adapter"   # a seed's epoch-2 adapter
RUNS = [                              # label, adapters (folders under checkpoints/), share of their averaged weight change;
                                      # "benchmark": True also scores that run on the test split
    {"label": "base", "adapters": [], "alpha": 1.0},   # untrained Qwen3.5-4B, served as published
    *({"label": f"seed{s}", "alpha": 1.0, "adapters": [RUN.format(seed=s)]} for s in SEEDS),
    {"label": "soup", "alpha": 1.0, "adapters": [RUN.format(seed=s) for s in SEEDS]},
    {"label": "soup_a075", "alpha": 0.75, "benchmark": True,             # the final model
     "adapters": [RUN.format(seed=s) for s in SEEDS]},
]
SETS_TO_SCORE = ["bench_train"]       # the benchmark's train split (never trained on), to compare the runs
ONLY = []                             # the labels to score in this pass ([] = every run)
BENCHMARK = True                      # load the test split, for the runs marked "benchmark": True only
SERVE_FLAGS = "--max-model-len 8192 --max-num-seqs 8 --logprobs-mode raw_logprobs"   # the benchmark scorer's defaults
CKPT_REPO = None                      # a private model repo: on Colab checkpoints/ is fetched from it and results pushed back
PUBLISH = False                       # after the table: rebuild the final model, check it on the test split (every
                                      # prediction kept) and upload it with its adapters to MODEL_REPO (private)
PUBLISH_RUN = "soup_a075"             # the final model, chosen on the train split
MODEL_REPO = "skamalj/signpost-d-4b"
RUN_NAME = "02_evaluate"


## Packages

The notebook itself needs pinned `transformers` (to build the model) and nothing on the GPU. vLLM goes into its own virtual environment (`vllm-env`, installed with `uv`), as the benchmark installs it: the current nightly. It runs as a separate server process, so its own `torch` and `transformers` never clash with the notebook's.

In [ ]:
!pip install -q --force-reinstall --no-deps "transformers==5.17.0"
!pip install -q uv

import gc, getpass, importlib.util, json, math, os, random, re, shutil, statistics, subprocess, sys, time
from collections import Counter
from pathlib import Path

import pandas as pd
import requests
import torch
import transformers
from huggingface_hub import HfApi, hf_hub_download, snapshot_download

VLLM_ENV = Path("vllm-env").absolute()
if not (VLLM_ENV / "bin" / "vllm").exists():
    subprocess.run([sys.executable, "-m", "uv", "venv", "-q", str(VLLM_ENV), "--python", sys.executable], check=True)
    subprocess.run([sys.executable, "-m", "uv", "pip", "install", "-q", "--python", str(VLLM_ENV / "bin" / "python"),
                    "-U", "vllm", "--torch-backend=auto", "--extra-index-url", "https://wheels.vllm.ai/nightly"], check=True)
VLLM_VERSION = subprocess.run([str(VLLM_ENV / "bin" / "python"), "-c", "import vllm; print(vllm.__version__)"],
                              capture_output=True, text=True).stdout.strip()
print(f"vLLM {VLLM_VERSION} in {VLLM_ENV}")

## The Hugging Face token

Read from the environment, a Kaggle secret or a Colab secret, and on Colab, if none is set, asked for in a hidden prompt; only where it came from is printed. The vLLM server inherits it from the environment.

In [ ]:
ON_COLAB = importlib.util.find_spec("google.colab") is not None


def hf_token():
    """-> (token or None, where it came from): the environment, a Kaggle secret, a Colab secret, or (on Colab) a prompt."""
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"], "the environment"
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN"), "Kaggle secret 'HF_TOKEN'"
    except Exception:
        pass
    if ON_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
            if token:
                return token, "Colab secret 'HF_TOKEN'"
        except Exception:
            pass
        token = getpass.getpass("Hugging Face token: ").strip()
        return token or None, "the prompt"
    return None, "NOT FOUND"


HF_TOKEN, token_from = hf_token()
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
print(f"Hugging Face token: {token_from}")          # where it came from, never the token itself

## Checkpoints and progress log

The trained adapters are in `checkpoints/` (with `CKPT_REPO`, downloaded from there on a fresh Colab machine). Results are written to `checkpoints/evaluation-vllm/` and pushed back. The notebook never starts CUDA itself: the whole GPU is left to the vLLM server.


In [ ]:
MAX_PROMPT_TOKENS = 8191                            # one question; vLLM serves 8,192 tokens including the answer
BENCH = "LocalLLaMA/typed-decisions"
PORT = 8000
SERVER = f"http://127.0.0.1:{PORT}"
CKPT = Path("checkpoints")
CKPT.mkdir(exist_ok=True)
if ON_COLAB and CKPT_REPO and not any(CKPT.iterdir()):   # a fresh Colab machine: fetch the trained runs
    snapshot_download(CKPT_REPO, repo_type="model", local_dir=str(CKPT), token=HF_TOKEN)
    print(f"checkpoints fetched from {CKPT_REPO}")
RUN_LOG = Path(f"{RUN_NAME}_run.log")
assert transformers.__version__ == "5.17.0", "restart the kernel and run from the top"
api = HfApi()
GPU_NAME = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()


def log(message):
    line = f"[{time.strftime('%H:%M:%S')}] {message}"
    print(line, flush=True)
    with open(RUN_LOG, "a") as fh:
        fh.write(line + "\n")


def sync_checkpoints(note):
    """On Colab, push checkpoints/ (with the new results) to CKPT_REPO. Never stops the notebook."""
    if not (ON_COLAB and CKPT_REPO):
        return
    try:
        api.upload_folder(folder_path=str(CKPT), repo_id=CKPT_REPO, repo_type="model", ignore_patterns=["*.tmp"],
                          commit_message=note, token=HF_TOKEN)
        log(f"  results pushed to {CKPT_REPO} ({note})")
    except Exception as e:
        log(f"  WARNING: push failed ({type(e).__name__}: {str(e)[:120]}); the local copy is intact")


log(f"{RUN_NAME} | {MODEL['name']} | runs {[r['label'] for r in RUNS]} | sets {SETS_TO_SCORE} | "
    f"benchmark {BENCHMARK} | vLLM {VLLM_VERSION} | {GPU_NAME}")

## The data

Our dev records come from `DATASET` (for the prompt check only), the benchmark's splits from `LocalLLaMA/typed-decisions`. Questions are rendered as in `01_train.ipynb` (vLLM's order and names: true/false as `yes` / `no`, choice names as written, score levels lowest first). The cases are sent to vLLM as they are; the items are used only to count the tokens of the prompt `01_train.ipynb` trains on.


In [ ]:
def load_jsonl(path):
    return [json.loads(line) for line in open(path, encoding="utf-8")]


def dataset_split(split):
    """One split of DATASET (train or dev) as a list of records."""
    return load_jsonl(hf_hub_download(DATASET, f"data/{split}.jsonl", repo_type="dataset", token=HF_TOKEN))


def render(q):
    """A System One question -> ({option name: description or None} in vLLM's order, the gold's keys in the same order):
    true/false as yes / no, choice by its names as written, score by its level strings, lowest first ("0".."n-1")."""
    criteria = q.get("criteria")
    if q["type"] == "noul":
        criteria = criteria or {}
        options = {"yes": criteria.get("true"), "no": criteria.get("false")}
        keys = ["true", "false"]
    elif q["type"] == "choice":
        options = {str(k): None if v is None else str(v) for k, v in criteria.items()}
        keys = list(criteria)
    else:
        options = {str(level): None for level in criteria}
        keys = [str(i) for i in range(len(criteria))]
    return options, keys


def items_of(case):
    """A case -> one item per question: the state, the question and its options (what the prompt is built from)."""
    state = case["state"] if isinstance(case["state"], str) else json.dumps(case["state"], ensure_ascii=False)
    out = []
    for qid, q in case["questions"].items():
        options, _ = render(q)
        out.append({"task": "d_" + q["type"], "id": case["id"], "qid": qid, "state": state,
                    "question": "" if q.get("instructions") is None else str(q["instructions"]), "options": options})
    return out

## The evaluation sets

- **bench_train:** the benchmark's `train` split (1,200 cases), never trained on: it compares the runs. Its questions are grouped by workflow and question.
- **benchmark** (only with `BENCHMARK`): the `test` split at its current revision, sent exactly as the benchmark scorer sends it, for the runs marked.


In [ ]:
DEV = dataset_split("dev")                          # our dev records: only for the prompt check
SETS = {}
if "bench_train" in SETS_TO_SCORE:                  # the benchmark's train split: never trained on
    path = hf_hub_download(BENCH, "all/train-00000-of-00001.parquet", repo_type="dataset")
    SETS["bench_train"] = [{"id": row["id"], "meta": {"kinds": {q: f"{row['workflow']}/{q}" for q in json.loads(row["questions"])}},
                            **{k: json.loads(row[k]) for k in ("state", "questions", "gold")}}
                           for row in pd.read_parquet(path).to_dict("records")]
KIND = {(c["id"], q): k for cs in SETS.values() for c in cs for q, k in c["meta"].get("kinds", {}).items()}

if BENCHMARK:
    revision = api.dataset_info(BENCH).sha
    path = hf_hub_download(BENCH, "all/test-00000-of-00001.parquet", repo_type="dataset", revision=revision)
    SETS["benchmark"] = [{"id": f"bench-{i}", **{k: json.loads(row[k]) for k in ("state", "questions", "gold")}}
                         for i, row in enumerate(pd.read_parquet(path).to_dict("records"))]
    log(f"benchmark test split @ {revision[:8]}: {len(SETS['benchmark'])} cases")
for name, cs in SETS.items():
    print(f"{name}: {len(cs)} cases | {dict(Counter(q['type'] for c in cs for q in c['questions'].values()))}")


## The prompt `01_train.ipynb` trains on

A copy of its prompt code: vLLM's wording, one question per sequence, the chat template with thinking off. Here it only counts tokens: for the first five dev records, the tokens it gives must equal the `input_tokens` vLLM reports. Equal counts mean the server reads the same prompt the model was trained on.

In [ ]:
LABELS = {"d_choice": [chr(65 + i) for i in range(26)],   # vLLM's labels, in option order
          "d_noul": ["yes", "no"],
          "d_score": [str(i) for i in range(10)]}
REPLY = {"d_choice": "Answer with the letter of one option only.",
         "d_noul": "Answer with yes or no only.",
         "d_score": "Answer with the number of one level only."}


def question_text(item):
    """The question as vLLM writes it after the state: the question, one line per labelled option, the reply line."""
    lines = [f"Question: {item['question']}"] if item["question"] else []
    for label, (name, description) in zip(LABELS[item["task"]], item["options"].items()):
        if item["task"] == "d_noul":                  # the side's description, or the bare label
            lines.append(f"{label}: {description}" if description else label)
        elif item["task"] == "d_choice":
            lines.append(f"{label}: {name} - {description}" if description else f"{label}: {name}")
        else:
            lines.append(f"{label}: {name}")
    lines.append(REPLY[item["task"]])
    return "\n".join(lines)


def user_turn(item):
    """vLLM's user message: the state, a blank line, the question."""
    return f"{item['state']}\n\n{question_text(item)}"


def question_ids(tok, item):
    """One question as vLLM sends it -> its token ids; the answer is the next token after the last one."""
    prompt = tok.apply_chat_template([{"role": "user", "content": user_turn(item)}], add_generation_prompt=True,
                                     tokenize=False, enable_thinking=False)
    return tok(prompt, add_special_tokens=False)["input_ids"]


from transformers import AutoModelForImageTextToText, AutoTokenizer

BASE_REV = api.model_info(MODEL["name"], token=HF_TOKEN).sha
tok = AutoTokenizer.from_pretrained(MODEL["name"], revision=BASE_REV, token=HF_TOKEN)
longest = max(len(question_ids(tok, it)) for cs in list(SETS.values()) + [DEV] for c in cs for it in items_of(c))
log(f"{MODEL['name']} @ {BASE_REV[:8]} | longest question prompt {longest} tokens (vLLM allows {MAX_PROMPT_TOKENS})")
assert longest <= MAX_PROMPT_TOKENS

## Building a run's model

LoRA adapters cannot be served by `/v1/systemone` as they are, so the weight changes are merged into the base, as `01_train.ipynb` builds the soup: alpha × the mean of each adapter's B·A × (lora_alpha / r), added to the matching weight. Training loaded only the text model, so its module names are matched to the full model by their layer and projection (`layers.<n>.<rest>`). The full model, vision tower and processor files included, is built on the CPU in fp16 (the precision the model was trained and scored in; bf16 would round away part of the small weight changes) and saved, so its architecture stays `Qwen3_5ForConditionalGeneration`.

In [ ]:
def layer_key(name):
    """'...layers.<n>.<rest>' -> 'layers.<n>.<rest>', the part both the text model and the full model share."""
    m = re.search(r"layers\.\d+\..+$", name)
    return m.group(0) if m else None


def export_model(run, out):
    """The base with alpha x the mean of the run's LoRA weight changes merged in, saved as a full fp16 Qwen3.5 checkpoint
    (vision tower, tokenizer and processor files included) in out."""
    from safetensors.torch import load_file
    if (out / "config.json").exists() and (out / "merged.json").exists():
        log(f"  model for {run['label']} already built in {out}")
        return
    shutil.rmtree(out, ignore_errors=True)
    snapshot_download(MODEL["name"], revision=BASE_REV, local_dir=str(out), token=HF_TOKEN,
                      ignore_patterns=["*.safetensors", "*.safetensors.index.json", "*.bin", "*.pt", "*.pth"])
    m = AutoModelForImageTextToText.from_pretrained(MODEL["name"], revision=BASE_REV, token=HF_TOKEN, dtype=torch.float16)
    assert type(m).__name__ == "Qwen3_5ForConditionalGeneration", type(m).__name__
    targets = {layer_key(n): mod for n, mod in m.named_modules()
               if "language_model.layers." in n and isinstance(mod, torch.nn.Linear)}
    dirs = [CKPT / a for a in run["adapters"]]
    merged = set()
    with torch.no_grad():
        for d in dirs:
            conf = json.load(open(d / "adapter_config.json"))
            scale = conf["lora_alpha"] / conf["r"] / len(dirs) * run["alpha"]
            weights = load_file(str(d / "adapter_model.safetensors"))
            for k, a in weights.items():
                if ".lora_A." not in k:
                    continue
                key = layer_key(k.split(".lora_A.")[0])
                assert key in targets, f"no module in the full model for {k}"
                w = targets[key].weight
                b = weights[k.replace(".lora_A.", ".lora_B.")].float()
                w += ((b @ a.float()) * scale).to(w.dtype)
                merged.add(key)
    m.save_pretrained(out)
    config = json.load(open(out / "config.json"))
    assert config["architectures"] == ["Qwen3_5ForConditionalGeneration"], config["architectures"]
    json.dump({"adapters": run["adapters"], "alpha": run["alpha"], "modules": len(merged), "base_revision": BASE_REV},
              open(out / "merged.json", "w"), indent=1)
    log(f"  built {run['label']}: {len(merged)} modules merged from {len(dirs)} adapter(s) at alpha {run['alpha']} -> {out}")
    del m
    gc.collect()

## Serving with vLLM

`vllm serve` with the benchmark scorer's flags, started from its own environment as a separate process, its output going to `vllm_<label>.log`. The notebook waits for `/health`; if the server stops, the end of its log is shown. vLLM picks the read strategy from the model's architecture; a Qwen model gets the generic one, one request per question.

In [ ]:
def start_server(model, label):
    """Starts vllm serve on model; returns the process once /health answers."""
    log_path = Path(f"vllm_{label}.log")
    used = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip()
    log(f"  GPU memory in use before vLLM starts: {used} (another kernel holding the GPU leaves vLLM too little)")
    proc = subprocess.Popen([str(VLLM_ENV / "bin" / "vllm"), "serve", str(model), "--port", str(PORT)] + SERVE_FLAGS.split(),
                            stdout=open(log_path, "w"), stderr=subprocess.STDOUT, env=os.environ.copy())
    t0 = time.time()
    while True:
        if proc.poll() is not None:
            lines = open(log_path).readlines()
            errors = list(dict.fromkeys(l.strip() for l in lines if "Error:" in l or ("ERROR" in l and "Traceback" not in l)))
            raise RuntimeError(f"vllm serve stopped (exit {proc.returncode}); its errors:\n" + "\n".join(errors[-12:])
                               + f"\n--- the end of {log_path}:\n" + "".join(lines[-15:]))
        try:
            if requests.get(SERVER + "/health", timeout=5).status_code == 200:
                break
        except requests.RequestException:
            pass
        if time.time() - t0 > 1800:
            proc.kill()
            raise RuntimeError(f"vllm serve not ready after 30 minutes; see {log_path}")
        time.sleep(10)
    log(f"  vLLM ready for {label} in {(time.time() - t0) / 60:.1f} min")
    return proc


def stop_server(proc):
    proc.terminate()
    try:
        proc.wait(timeout=120)
    except subprocess.TimeoutExpired:
        proc.kill()
    time.sleep(5)                                       # the GPU memory is released before the next server starts

## Asking a case

One `POST /v1/systemone` per case: the state and its questions exactly as stored (no `instructions`, no chat options), the same body the benchmark scorer sends. The reply has, per question, the probabilities (keyed `yes` / `no`, by option name, or by level index) and vLLM's diagnostics; the time is measured around the whole request.

In [ ]:
def ask(model_name, case):
    """-> (the response JSON, milliseconds for the whole case)."""
    payload = {"model": model_name, "state": case["state"], "questions": case["questions"]}
    t0 = time.perf_counter()
    r = requests.post(SERVER + "/v1/systemone", json=payload, timeout=300)
    ms = (time.perf_counter() - t0) * 1000
    if r.status_code != 200:
        raise RuntimeError(f"/v1/systemone answered {r.status_code} for case {case['id']}: {r.text[:500]}")
    return r.json(), ms

## The measures

`score_one` is copied unchanged from the benchmark's `bench/score_with_jobs.py`: `yes` / `no` become `true` / `false`, the predicted probabilities are renormalised over the gold's options (missing ones count as 0), then accuracy (top option = gold's top option), KL from the gold with a 1e-9 floor, Brier, and the top probability as the confidence for a 10-bin ECE. Results are given overall, per type and per kind of question, with 95% intervals from resampling cases for the `test` split.

In [ ]:
def score_one(pred, gold):                           # copied unchanged from the benchmark's bench/score_with_jobs.py
    opts = list(gold["probabilities"].keys())
    g = [float(gold["probabilities"][o]) for o in opts]
    if gold["type"] == "noul":  # engines report yes/no; gold uses true/false
        pred = dict(pred)
        pred["probabilities"] = {
            {"yes": "true", "no": "false"}.get(k, k): v for k, v in pred["probabilities"].items()
        }
    p = [float(pred["probabilities"].get(o, 0.0)) for o in opts]
    s = sum(p)
    p = [v / s for v in p] if s > 0 else [1.0 / len(opts)] * len(opts)
    am_p = p.index(max(p))
    am_g = g.index(max(g))
    kl = sum(gi * math.log(gi / max(pi, 1e-9)) for gi, pi in zip(g, p) if gi > 0)
    brier = sum((pi - gi) ** 2 for pi, gi in zip(p, g))
    return am_p == am_g, kl, brier, max(p)


def summary(rows):
    bins = [[0, 0, 0.0] for _ in range(10)]
    for r in rows:
        b = bins[min(int(r["conf"] * 10), 9)]
        b[0] += 1; b[1] += r["ok"]; b[2] += r["conf"]
    ece = sum(b[0] / len(rows) * abs(b[1] / b[0] - b[2] / b[0]) for b in bins if b[0])
    return {"n": len(rows), "accuracy": round(statistics.mean(r["ok"] for r in rows), 4),
            "kl": round(statistics.mean(r["kl"] for r in rows), 4),
            "brier": round(statistics.mean(r["brier"] for r in rows), 4), "ece": round(ece, 4)}


def report(rows, intervals=False):
    """Overall, per type, per kind and, with intervals, 95% intervals from resampling cases."""
    out = {"all": summary(rows), **{t: summary([r for r in rows if r["type"] == t]) for t in sorted({r["type"] for r in rows})}}
    kinds = sorted({r["kind"] for r in rows})
    if kinds != ["?"]:
        out["by_kind"] = {k: summary([r for r in rows if r["kind"] == k]) for k in kinds}
    if intervals:
        cases = {}
        for r in rows:
            cases.setdefault(r["id"], []).append(r)
        groups, rng, acc, kl = list(cases.values()), random.Random(0), [], []
        for _ in range(1000):
            sample = [r for g in (rng.choice(groups) for _ in groups) for r in g]
            acc.append(statistics.mean(r["ok"] for r in sample)); kl.append(statistics.mean(r["kl"] for r in sample))
        acc.sort(); kl.sort()
        out["ci95"] = {"accuracy": [round(acc[25], 4), round(acc[974], 4)], "kl": [round(kl[25], 4), round(kl[974], 4)]}
    return out

## Scoring a run

For each run: build its model (the base is served as published), start vLLM, check the prompt's token count against `01_train.ipynb`'s on the first five dev records, then score every set not scored before, one case at a time as the benchmark does. Each set's result is saved, so running the notebook again reads it back. The built model is deleted afterwards (about 9 GB).

In [ ]:
def prompt_check(served):
    """vLLM's input tokens for the first five dev cases against the training prompt's token count."""
    for case in DEV[:5]:
        resp, _ = ask(served, case)
        ours = sum(len(question_ids(tok, it)) for it in items_of(case))
        theirs = resp["usage"]["input_tokens"]
        log(f"  prompt check {case['id']}: training prompt {ours} tokens, vLLM {theirs} | {'MATCH' if ours == theirs else 'MISMATCH'}")


def score_set(served, name, rdir):
    rows, ms, mass, label_top, preds = [], [], [], [], []
    for case in SETS[name]:
        resp, t = ask(served, case)
        ms.append(t)
        for qid, gold in case["gold"].items():
            ok, kl, brier, conf = score_one(resp["answers"].get(qid, {"probabilities": {}}), gold)
            rows.append({"id": case["id"], "type": gold["type"], "kind": KIND.get((case["id"], qid), "?"),
                         "ok": ok, "kl": kl, "brier": brier, "conf": conf})
            diag = resp.get("diagnostics", {}).get(qid, {})
            mass.append(diag.get("label_mass", float("nan")))
            label_top.append(bool(diag.get("argmax_is_label")))
            preds.append({"id": case["id"], "qid": qid, "p": resp["answers"].get(qid, {}).get("probabilities", {}),
                          "label_mass": diag.get("label_mass")})
    if name != "benchmark":                          # every answer, for analysis (never kept for the test split)
        json.dump(preds, open(rdir / f"{name}_predictions.json", "w"))
    res = report(rows, intervals=name == "benchmark")
    ms.sort()
    res["vllm"] = {"label_mass_mean": round(statistics.mean(mass), 4), "argmax_is_label": round(statistics.mean(label_top), 4),
                   "ms_per_case_p50": round(ms[len(ms) // 2]), "ms_per_case_p95": round(ms[int(len(ms) * 0.95)]),
                   "gpu": GPU_NAME, "vllm": VLLM_VERSION}
    return res


def evaluate(run):
    rdir = CKPT / "evaluation-vllm" / run["label"]
    rdir.mkdir(parents=True, exist_ok=True)
    names = [name for name in SETS if name != "benchmark" or run.get("benchmark")]
    todo = [name for name in names if not (rdir / f"{name}.json").exists()]
    if todo:
        log(f"===== {run['label']}: {run} | to score: {todo}")
        out = None
        if run["adapters"]:
            out = Path("exports") / run["label"]
            export_model(run, out)
            served = str(out.absolute())
        else:
            served = MODEL["name"]
        proc = start_server(served, run["label"])
        try:
            prompt_check(served)
            for name in todo:
                t0 = time.perf_counter()
                res = score_set(served, name, rdir)
                res["minutes"] = round((time.perf_counter() - t0) / 60, 1)
                json.dump(res, open(rdir / f"{name}.json", "w"), indent=1)
                types = " ".join(f"{k} {v['accuracy']}" for k, v in res.items() if k in ("noul", "choice", "score"))
                kinds = " ".join(f"{k} {v['accuracy']}" for k, v in res.get("by_kind", {}).items())
                log(f"{run['label']} {name}: {res['all']} | {types} | {kinds} | {res['vllm']}"
                    + (f" | 95% {res['ci95']}" if "ci95" in res else ""))
        finally:
            stop_server(proc)
        sync_checkpoints(f"vLLM evaluation {run['label']}: {todo}")
        if out:
            shutil.rmtree(out, ignore_errors=True)
    else:
        log(f"===== {run['label']}: every set scored earlier, read back")
    return {name: json.load(open(rdir / f"{name}.json")) for name in names}

## Run, and the table

Scores every run in `RUNS` and prints accuracy and KL per set, then vLLM's label mass (near 1 means the model puts its probability on the labels, as it should when answering in this format) and the median time per case.

In [ ]:
ready = [run for run in RUNS if not ONLY or run["label"] in ONLY   # ONLY limits this pass; untrained runs are skipped
         if all((CKPT / a / "adapter_model.safetensors").exists() for a in run["adapters"])]
log(f"runs to score: {[r['label'] for r in ready]} | skipped (not trained yet): {[r['label'] for r in RUNS if r not in ready]}")
RESULTS = {run["label"]: evaluate(run) for run in ready}
cell = lambda res, n, f: f(res[n]) if n in res else f"{'-':>22s}"
print(f"\n{'run':14s} " + " ".join(f"{name:>22s}" for name in SETS))
for label, res in RESULTS.items():
    print(f"{label:14s} " + " ".join(cell(res, n, lambda r: f"{r['all']['accuracy']:>10.4f} / KL {r['all']['kl']:.3f}") for n in SETS))
print(f"\n{'run':14s} " + " ".join(f"{name:>22s}" for name in SETS) + "   (label mass | ms per case, median)")
for label, res in RESULTS.items():
    print(f"{label:14s} " + " ".join(cell(res, n, lambda r: f"{r['vllm']['label_mass_mean']:>10.3f} | {r['vllm']['ms_per_case_p50']:>7d} ms") for n in SETS))
for label, res in RESULTS.items():
    if "benchmark" in res:
        b = res["benchmark"]
        print(f"\nbenchmark, {label}: {b['all']} | 95% {b['ci95']} | "
              + " ".join(f"{t} {b[t]['accuracy']}" for t in ("noul", "choice", "score") if t in b))

## Publishing the final model

With `PUBLISH = True`: the final model is built again from its three adapters, served, and scored on the `test` split once more, this time keeping every prediction (the passes above keep none for the `test` split). Then the model, the three adapters (`adapters/seed42`, `seed1042`, `seed2042`) and the predictions (`typed_decisions_test_predictions.json`) are uploaded to `MODEL_REPO`, created private. The card and the eval results are added separately.


In [ ]:
if PUBLISH:
    run = next(r for r in RUNS if r["label"] == PUBLISH_RUN)
    out = Path("exports") / f"publish-{run['label']}"
    export_model(run, out)
    for a in run["adapters"]:                       # each seed's epoch-2 adapter, so the soup can be rebuilt
        seed = re.search(r"attempt-seed(\d+)", a).group(1)
        (out / "adapters" / f"seed{seed}").mkdir(parents=True, exist_ok=True)
        for f in ("adapter_config.json", "adapter_model.safetensors"):
            shutil.copy(CKPT / a / f, out / "adapters" / f"seed{seed}" / f)
    served = str(out.absolute())
    proc = start_server(served, run["label"])
    try:
        prompt_check(served)
        rows, preds = [], []
        for i, case in enumerate(SETS["benchmark"]):
            resp, _ = ask(served, case)
            for qid, gold in case["gold"].items():
                pred = resp["answers"].get(qid, {"probabilities": {}})
                ok, kl, brier, conf = score_one(pred, gold)
                rows.append({"id": case["id"], "type": gold["type"], "kind": "?", "ok": ok, "kl": kl, "brier": brier, "conf": conf})
                preds.append({"case": i, "question": qid, "type": gold["type"], "probabilities": pred["probabilities"]})
    finally:
        stop_server(proc)
    check = report(rows, intervals=True)
    earlier = json.load(open(CKPT / "evaluation-vllm" / run["label"] / "benchmark.json"))["all"]
    log(f"publish check, test split: {check['all']} | 95% {check['ci95']} | earlier {earlier}")
    json.dump(preds, open(out / "typed_decisions_test_predictions.json", "w"))
    api.create_repo(MODEL_REPO, private=True, exist_ok=True, token=HF_TOKEN)
    api.upload_folder(folder_path=str(out), repo_id=MODEL_REPO, token=HF_TOKEN,
                      commit_message=f"Signpost-D 4B: merged model ({run['label']}), three seed adapters, test predictions")
    log(f"uploaded {out} to {MODEL_REPO} (private): {sorted(f.rfilename for f in api.model_info(MODEL_REPO, token=HF_TOKEN).siblings)}")
